In [ ]:
import os
import polars as pl
import tomllib
from pathlib import Path
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import pandas as pd

config_file_path = Path("src/config.toml")

with config_file_path.open("rb") as config_file:
    config = tomllib.load(config_file)

MODELED_PARQUET_DIR = os.path.join(config['GOLD_DIR'], 'parquet')
COMPANIES = config['companies']

company = next(company for company in COMPANIES)

lf = pl.scan_parquet(os.path.join(MODELED_PARQUET_DIR, f'{company}_share_prices_modeled.parquet'))

target_classification = 'Target Return Class'
target_regression = 'Target Return Metric'

feature_columns = [
    column 
    for column in lf.collect_schema().names()
    if column not in {target_classification, target_regression}
]

classification_lf = (
    lf
    .select(feature_columns + [target_classification])
    .with_columns(pl.col(target_classification).cast(pl.Int64))
)

classification_df = classification_lf.collect().to_pandas()

X = classification_df[feature_columns]
y = classification_df[target_classification]

test_fraction = 0.20
test_size = int(len(classification_df) * test_fraction)

X_train = X.iloc[:-test_size].copy()
y_train = y.iloc[:-test_size].copy()

X_test = X.iloc[-test_size:].copy()
y_test = y.iloc[-test_size:].copy()

time_series_split = TimeSeriesSplit(
    n_splits=5,
    test_size=max(1, len(X_train) // 10),
    gap=0,
)

xgb_classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
)

parameter_distributions = {
    "n_estimators": [200, 400, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0.0, 0.1, 0.3, 1.0],
    "reg_alpha": [0.0, 0.01, 0.1, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

hyperparameter_search = RandomizedSearchCV(
    estimator=xgb_classifier,
    param_distributions=parameter_distributions,
    n_iter=40,
    scoring="roc_auc",
    cv=time_series_split,
    verbose=1,
    n_jobs=-1,
    random_state=42,
    refit=True,
)

hyperparameter_search.fit(X_train, y_train)

print(hyperparameter_search.best_score_)
print(hyperparameter_search.best_params_)

best_model = hyperparameter_search.best_estimator_

test_probabilities = best_model.predict_proba(X_test)[:, 1]
test_predictions = best_model.predict(X_test)

print("Test ROC AUC:", roc_auc_score(y_test, test_probabilities))
print("Test Accuracy:", accuracy_score(y_test, test_predictions))
print(classification_report(y_test, test_predictions))